In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from transformers import AutoFeatureExtractor, ViTForImageClassification
from torch.cuda.amp import autocast, GradScaler

# pick one device abeg
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ===============================
# 1. Hyperparameters
# ===============================
num_epochs = 10
batch_size = 16
learning_rate = 3e-5
num_classes = 4
val_split = 0.2  # 20% of train for validation

# ===============================
# 2. Data Loading and Preprocessing
# ===============================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Training'
test_path = 'C:/Users/JARE WORKS/Documents/aj project/ovie results/archive (3)/Testing'

full_train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
test_dataset = datasets.ImageFolder(root=test_path, transform=transform)

# split into train and val
torch.manual_seed(42)
val_size = int(len(full_train_dataset) * val_split)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# ===============================
# 3. Vision Transformer Model
# ===============================
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch32-224-in21k')
vit_model.heads.head = nn.Linear(vit_model.heads.head.in_features, num_classes)
vit_model = vit_model.to(device)

# ===============================
# 4. Loss and Optimizer
# ===============================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(vit_model.parameters(), lr=learning_rate)

# ===============================
# 5. Training with Validation Loop
# ===============================
def train_with_val(model, train_loader, val_loader, criterion, optimizer, epochs):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        avg_train_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(avg_train_loss)

        # ---- Validation phase ----
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)

                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct / total

        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {avg_train_loss:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_vit_model.pth')

    return history

# ===============================
# 6. Test Evaluation
# ===============================
def evaluate_model(model, loader, class_names):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    conf_mat = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    plt.imshow(conf_mat, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45)
    plt.yticks(np.arange(len(class_names)), class_names)
    plt.tight_layout()
    plt.show()

# ===============================
# 7. Run Training and Evaluation
# ===============================
start = time.time()
history = train_with_val(vit_model, train_loader, val_loader, criterion, optimizer, num_epochs)
print(f"Training done in {time.time() - start:.2f} seconds")

# Load best model and evaluate on test
vit_model.load_state_dict(torch.load('best_vit_model.pth'))
evaluate_model(vit_model, test_loader, full_train_dataset.classes)

# ===============================
# 8. Plot Training and Validation Curves
# ===============================
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Over Epochs")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Over Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


c:\Users\JARE WORKS\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch32-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'vit_model' is not defined

In [ ]:
# ADD AT THE END OF YOUR EXISTING CODE

from collections import Counter
import seaborn as sns
import pandas as pd

# ===============================
# 9. Plot Training and Validation Curves
# ===============================
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs. Validation Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], label='Validation Accuracy', color='green')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Over Epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ===============================
# 10. Class Distribution in Train & Val
# ===============================
def plot_class_distribution(dataset, label, class_names):
    counts = Counter([dataset[i][1] for i in range(len(dataset))])
    class_counts = [counts[i] for i in range(len(class_names))]

    plt.figure(figsize=(6, 4))
    sns.barplot(x=class_names, y=class_counts)
    plt.title(f"Class Distribution in {label} Set")
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_class_distribution(train_dataset, "Train", full_train_dataset.classes)
plot_class_distribution(val_dataset, "Validation", full_train_dataset.classes)

# ===============================
# 11. Confusion Matrix with Annotations
# ===============================
def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()

# get preds and true labels again for detailed plots
def get_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

preds, true_labels = get_predictions(vit_model, test_loader)
cm = confusion_matrix(true_labels, preds)
plot_confusion_matrix(cm, full_train_dataset.classes)

# ===============================
# 12. Classification Report Bar Graphs
# ===============================
def plot_classification_report(true_labels, preds, class_names):
    report = classification_report(true_labels, preds, target_names=class_names, output_dict=True)
    df = pd.DataFrame(report).transpose()

    metrics = ['precision', 'recall', 'f1-score']
    df = df.loc[class_names]  # Keep only class rows

    df[metrics].plot(kind='bar', figsize=(10, 6))
    plt.title("Per-Class Precision, Recall, and F1-score")
    plt.ylabel("Score")
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_classification_report(true_labels, preds, full_train_dataset.classes)

# ===============================
# 13. Correct vs Incorrect Predictions (Pie Chart)
# ===============================
def plot_correct_vs_incorrect(true_labels, preds):
    correct = (true_labels == preds).sum()
    incorrect = (true_labels != preds).sum()
    labels = ['Correct', 'Incorrect']
    sizes = [correct, incorrect]

    plt.figure(figsize=(5, 5))
    plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['#66bb6a', '#ef5350'])
    plt.title("Prediction Accuracy Breakdown")
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

plot_correct_vs_incorrect(true_labels, preds)
